In [1]:
'''
This Capstone project has been done using Movie_Collection.csv dataset. By providing budget, trailer_views, marketing_expenses, etc.,
input features, we can predict the collection.
'''

'\nThis Capstone project has been done using Movie_Collection.csv dataset. By providing budget, trailer_views, marketing_expenses, etc.,\ninput features, we can predict the collection.\n'

In [2]:
# downloading important libraries
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt


In [3]:
# Function: SelectKBest (Chi-square (chi² test))
# (It selects the top n most important features from the dataset using a statistical test)
def selectkbest(indep_X, dep_Y, n):
#indep_X => features/input variables; dep_Y => target/output variables; n =>  Number of top features to select
    from sklearn.feature_selection import SelectKBest, chi2
    
    selector = SelectKBest(score_func=chi2, k=n) # Using Chi-square Technique select the top n features
    selector.fit(indep_X, dep_Y) # Each feature tested against the target and a high score feature is considered as a important feature

    # Get selected columns
    selected_columns = indep_X.columns[selector.get_support()]

    # Transform dataset
    X_new = selector.transform(indep_X)

    return X_new, selected_columns, selector

In [4]:
# Function
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()  # to treat/convert all input varialbes/features with in range
        X_train = sc.fit_transform(X_train)  # It learns the scale of your training data and converts it into standardized form (mean = 0, std = 1)
        X_test = sc.transform(X_test) # for testing fit_transform as no conversion required for test data and here only testing on raw data   
        return X_train, X_test, y_train, y_test, sc

In [5]:
# Function
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test) # for the X_test input predicting the output and storing into y_pred
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred) # comparison between actual output and predicted output
     return r2

In [6]:
# Function
def random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train) # Model Creation
        r2=r2_prediction(regressor,X_test,y_test) # Prediction
        return  regressor,r2 

In [7]:
# Function
def selectk_regression(accrf): 
    
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Random Forest'])

    for number,idex in enumerate(dataframe.index):  # enumerate used to have Row position Along with actual index label
        dataframe['Random Forest'][idex]=accrf[number]
    return dataframe

In [8]:
# DATA COLLECTION
dataset1=pd.read_csv("Movie_Collection.csv",index_col=None)



In [9]:
dataset1

,Marketing_expense,Production_expense,Multiplex_coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection,Start_Tech_Oscar
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000,1
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200,0
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400,1
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800,1
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800,0
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200,0
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800,0
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000,0


In [10]:
# DATA PREPROCESSING

df2=dataset1

df2 = pd.get_dummies(df2, drop_first=True)
df2.isna().values.any() # To confirm any NaN values exists in the dataset
df2.isna().sum() #Finding for which column how many NaN values are exist
df2['Time_taken'] = df2['Time_taken'].fillna(0) # Replacing the NaN value with 0
df2.isna().values.any() # To confirm any NaN values exists in the dataset
df2.isna().sum() #Finding for which column how many NaN values are exist
df2

,Marketing_expense,Production_expense,Multiplex_coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,...,Time_taken,Twitter_hastags,Avg_age_actors,Num_multiplex,Collection,Start_Tech_Oscar,3D_available_YES,Genre_Comedy,Genre_Drama,Genre_Thriller
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,...,109.60,223.840,23,494,48000,1,1,0,0,1
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,...,146.64,243.456,42,462,43200,0,0,0,1,0
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,...,147.88,2022.400,38,458,69400,1,0,1,0,0
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,...,185.36,225.344,45,472,66800,1,1,0,1,0
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,...,176.48,225.792,55,395,72400,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,...,186.96,243.584,27,561,44800,0,0,0,0,0
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,...,132.24,263.296,20,600,41200,0,1,0,0,0
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,...,109.56,243.824,31,576,47800,0,0,1,0,0
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,...,158.80,303.520,47,607,44000,0,1,1,0,0


In [11]:
#Splitting input and output features
indep_X=df2.drop('Collection', 1) # removing the output column 'Collection' from the input set
dep_Y=df2['Collection'] # keeping only the output column 'Collection' in the output set

In [12]:
indep_X
indep_X.isna().sum() # Confirming that no NaN values exist in the dataset

Marketing_expense      0
Production_expense     0
Multiplex_coverage     0
Budget                 0
Movie_length           0
Lead_ Actor_Rating     0
Lead_Actress_rating    0
Director_rating        0
Producer_rating        0
Critic_rating          0
Trailer_views          0
Time_taken             0
Twitter_hastags        0
Avg_age_actors         0
Num_multiplex          0
Start_Tech_Oscar       0
3D_available_YES       0
Genre_Comedy           0
Genre_Drama            0
Genre_Thriller         0
dtype: int64

In [13]:
dep_Y
dep_Y.isna().sum() # Confirming that no NaN values exist in the dataset

0

In [14]:
# FEATURE SELECTION

#kbest=selectkbest(indep_X,dep_Y,5)   
kbest, selected_columns, selector = selectkbest(indep_X, dep_Y, 5)

print("Selected Features:", selected_columns)

accrf=[]

Selected Features: Index(['Marketing_expense', 'Budget', 'Trailer_views', 'Twitter_hastags',
       'Num_multiplex'],
      dtype='object')


In [15]:
kbest

array([[2.0126400e+01, 3.6524125e+04, 5.2736700e+05, 2.2384000e+02,
        4.9400000e+02],
       [2.0546200e+01, 3.5668655e+04, 4.9405500e+05, 2.4345600e+02,
        4.6200000e+02],
       [2.0545800e+01, 3.9912675e+04, 5.4705100e+05, 2.0224000e+03,
        4.5800000e+02],
       ...,
       [2.1215200e+01, 3.8751680e+04, 5.3223900e+05, 2.4382400e+02,
        5.7600000e+02],
       [2.2191800e+01, 3.7740670e+04, 4.9607700e+05, 3.0352000e+02,
        6.0700000e+02],
       [2.0948200e+01, 3.3496650e+04, 5.1843800e+05, 2.0304000e+02,
        6.0400000e+02]])

In [16]:
X_train, X_test, y_train, y_test, sc=split_scalar(kbest,dep_Y) # Keeping only the best input variables/features and splitting the test and train data 
for i in kbest:   
      
    regressor, r2_r=random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=selectk_regression(accrf)

In [17]:
result # KBest as 5

,Random Forest
ChiSquare,0.692255


# SAVE THE BEST MODEL

In [18]:
import pickle # download the pickle library to save the model

model_data = {
    "model": regressor,
    "columns": selected_columns,
    "scaler": sc   # IMPORTANT
}

pickle.dump(model_data, open("Finalized_Model_RandomForest.sav", "wb"))
#filename = "Finalized_Model_RandomForest.sav" # create a sav file to save our model
#pickle.dump(regressor,open(filename,'wb')) # our model has been copying into the file, can see the .sav file in our working dir

# Test our model

In [19]:
#loaded_model=pickle.load(open("Finalized_Model_RandomForest.sav","rb")) # loading our model into a variable
#result=loaded_model.predict([[130298.13,145530.06,323876.68,1,0]])
#result

In [20]:
import pickle
import pandas as pd

# Load everything
model_data = pickle.load(open("Finalized_Model_RandomForest.sav", "rb"))

model = model_data["model"]
columns = model_data["columns"]
scaler = model_data["scaler"]

# --- USER INPUT ---
input_dict = {}

for col in columns:
    value = float(input(f"Enter value for {col}: "))
    input_dict[col] = value

# Convert to DataFrame
input_df = pd.DataFrame([input_dict])

# Scale
input_scaled = scaler.transform(input_df)

# Predict
result = model.predict(input_scaled)
print("********************")
print("Predicted Collection:", result[0])
print("********************")


Enter value for Marketing_expense: 50000
Enter value for Budget: 2000
Enter value for Trailer_views: 10
Enter value for Twitter_hastags: 2
Enter value for Num_multiplex: 89
********************
Predicted Collection: 21140.0
********************


C:\Users\krish\Anaconda3\envs\krish_virtual\lib\site-packages\sklearn\base.py:444: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  f"X has feature names, but {self.__class__.__name__} was fitted without"
